# Dataset for the adaptive immune receptor chapters

The adaptive immune receptor chapters use peripheral blood mononuclear cells from the COVID-19 study of Stephenson et al., for which gene expression, TCR and BCR libraries were sequenced from the same cells.
We keep the samples of the Cambridge site at the day of recruitment, which cover healthy donors and all degrees of COVID-19 severity, with one sample per donor.
Donor CV0198 is excluded because almost all of their cells belong to a malignant B cell clone.
We further store the human TCRs of VDJdb and the antibodies of CoV-AbDab, against which the specificity chapter queries the receptors.

In [1]:
import lamindb as ln

ln.connect("theislab/sc-best-practices")

ln.track("7VnegJfx1z06")

→ connected lamindb: theislab/sc-best-practices


! It looks like you are running jupyter lab but don't have jupyter-server module installed. Please install it via pip install jupyter-server


→ created Transform('7VnegJfx1z060000', key='air_repertoire_dataset.ipynb'), started new Run('iiHFBP62yyV8t1Aj') at 2026-09-29 16:55:52 UTC


→ notebook imports: anndata==0.13.2 awkward==2.14.0 lamindb-core==2.10.0 mudata==0.4.1 pandas==3.0.6 pooch==1.9.0 scirpy==0.25.1


In [2]:
import tempfile
from pathlib import Path

import anndata as ad
import awkward as ak
import mudata as md
import pandas as pd
import pooch
import scirpy as ir

The gene expression, provided with raw counts, and the TCR contigs come from ArrayExpress (E-MTAB-10026).
The BCR contigs were re-annotated with IgBLAST and their germline sequences reconstructed by the Clatworthy lab, who analysed the B cells of the study.

In [3]:
arrayexpress = "https://ftp.ebi.ac.uk/biostudies/fire/E-MTAB-/026/E-MTAB-10026/Files"
gex_path = pooch.retrieve(
    f"{arrayexpress}/covid_portal_210320_with_raw.h5ad",
    known_hash="sha256:ec48f328f2e884c23376c8aa1f26041e11625762be5c30b0bd0869aa8bb1a334",
)
tcr_path = pooch.retrieve(
    f"{arrayexpress}/TCR_merged-Updated.tsv",
    known_hash="sha256:e36d4a7527454e832747dcd2f5360c6f37bfd6b7e1251f197e59eda1066da407",
)
bcr_path = pooch.retrieve(
    "https://raw.githubusercontent.com/clatworthylab/COVID_analysis/e492c848daa48d24fe7cc872aab1027495d2ac43/scbcr_airr.tsv.bz2",
    known_hash="sha256:ad891d2bfc90b39ba4cb15029ffb02e785cd43d8be0b55052ca9413f80a1c1b0",
)
cov_abdab_path = pooch.retrieve(
    "https://opig.stats.ox.ac.uk/webapps/covabdab/static/downloads/CoV-AbDab_080224.csv",
    known_hash="sha256:fdc7e649e8b2e72902ddd7080e8c81e8a8af1ae6afda37913dc578b229763b96",
)

## Gene expression

In [4]:
gex = ad.read_h5ad(gex_path, backed="r")
keep = (
    (gex.obs["Site"] == "Cambridge")
    & (gex.obs["Collection_Day"] == "D0")
    & (gex.obs["patient_id"] != "CV0198")
    & (gex.obs["sample_id"] != "BGCV01_CV0902")
)
genes = gex.var["feature_types"] == "Gene Expression"
gex = gex[keep.to_numpy(), genes.to_numpy()].to_memory()
gex.X = gex.layers.pop("raw")
gex.obs = gex.obs[
    [
        "patient_id",
        "sample_id",
        "Sex",
        "Age_interval",
        "Status_on_day_collection_summary",
        "Days_from_onset",
        "Worst_Clinical_Status",
        "Outcome",
        "initial_clustering",
        "full_clustering",
    ]
].rename(
    columns={
        "Sex": "sex",
        "Age_interval": "age",
        "Status_on_day_collection_summary": "severity",
        "Days_from_onset": "days_from_onset",
        "Worst_Clinical_Status": "worst_clinical_status",
        "Outcome": "outcome",
        "initial_clustering": "cell_type_coarse",
        "full_clustering": "cell_type",
    }
)
gex.obs = gex.obs.apply(
    lambda col: col.cat.remove_unused_categories() if col.dtype == "category" else col
)
gex.var = gex.var.drop(columns="feature_types")
del gex.obsm["X_pca"]
gex.uns = {}
gex

AnnData object with n_obs × n_vars = 104492 × 24737
    obs: 'patient_id', 'sample_id', 'sex', 'age', 'severity', 'days_from_onset', 'worst_clinical_status', 'outcome', 'cell_type_coarse', 'cell_type'
    obsm: 'X_pca_harmony', 'X_umap'
    layers: None (.X)

## Immune receptors

The TCR contigs are in the 10x Genomics format, but identify cells by the `CellID` column, which matches the cell names of the gene expression.
Both receptor tables are subset to the selected cells before scirpy reads them.

In [5]:
with tempfile.TemporaryDirectory() as tmp:
    tcr = pd.read_csv(tcr_path, sep="\t", low_memory=False)
    tcr = tcr[tcr["CellID"].isin(gex.obs_names)]
    tcr["barcode"] = tcr.pop("CellID")
    tcr.to_csv(Path(tmp) / "tcr.csv", index=False)
    adata_tcr = ir.io.read_10x_vdj(Path(tmp) / "tcr.csv")

    bcr = pd.read_csv(bcr_path, sep="\t", low_memory=False, index_col=0)
    bcr = bcr[bcr["cell_id"].isin(gex.obs_names)]
    bcr.to_csv(Path(tmp) / "bcr.tsv", sep="\t", index=False)
    adata_bcr = ir.io.read_airr(Path(tmp) / "bcr.tsv")

airr = ir.pp.merge_airr(adata_tcr, adata_bcr)
mdata = md.MuData({"gex": gex, "airr": airr})
mdata

/home/lheumos/miniforge3/envs/air-repertoire/lib/python3.13/site-packages/scirpy/io/_convert_anndata.py:52: ExperimentalFeatureWarning: Support for Awkward Arrays is currently experimental. Behavior may change in the future. Please report any issues you may encounter!
  adata = AnnData(


  0%|          | 0/13 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

MuData object with n_obs × n_vars = 104492 × 24737
  2 modalities
    gex:	104492 × 24737
      obs:	'patient_id', 'sample_id', 'sex', 'age', 'severity', 'days_from_onset', 'worst_clinical_status', 'outcome', 'cell_type_coarse', 'cell_type'
      obsm:	'X_pca_harmony', 'X_umap'
      layers:	None
    airr:	75188 × 0
      uns:	'scirpy_version'
      obsm:	'airr'

In [6]:
ln.Artifact.from_mudata(
    mdata,
    key="air_repertoire/stephenson2021_cambridge.h5mu",
    description="Stephenson 2021 COVID-19 PBMCs of the Cambridge site with gene expression, TCRs and BCRs",
).save()

... uploading n4YVerQtkACr5QBS0000.h5mu:  0.0%

... uploading n4YVerQtkACr5QBS0000.h5mu:  3.4%

... uploading n4YVerQtkACr5QBS0000.h5mu: 10.2%

... uploading n4YVerQtkACr5QBS0000.h5mu: 13.7%

... uploading n4YVerQtkACr5QBS0000.h5mu: 17.1%

...

... uploading n4YVerQtkACr5QBS0000.h5mu: 58.0%

... uploading n4YVerQtkACr5QBS0000.h5mu: 69.3%

... uploading n4YVerQtkACr5QBS0000.h5mu: 72.7%

... uploading n4YVerQtkACr5QBS0000.h5mu: 100.0%


Artifact(uid='n4YVerQtkACr5QBS0000', key='air_repertoire/stephenson2021_cambridge.h5mu', description='Stephenson 2021 COVID-19 PBMCs of the Cambridge site with gene expression, TCRs and BCRs', suffix='.h5mu', kind='dataset', otype='MuData', size=1536325215, hash='0_JeXUUtdUhNVuFJk2-uOO', n_files=None, n_observations=104492, extra_data=None, branch_id=1, created_on_id=1, space_id=1, storage_id=1, run_id=175, schema_id=None, created_by_id=2, created_at=2026-09-29 16:57:57 UTC, is_locked=False, version_tag=None, is_latest=True)

## Reference databases

We take VDJdb from a pinned release of IggyTop, which harmonizes the CDR3 sequences of several databases to AIRR junctions.
VDJdb reports V and J genes with alleles, whereas Cell Ranger reports them without, so we remove the alleles to be able to require matching V genes.

In [7]:
vdjdb = ir.datasets.vdjdb(tag="data-2026.09.28.152145")
vdjdb = vdjdb[vdjdb.obs["source_organism"] == "Homo sapiens"].copy()
chains = vdjdb.obsm["airr"]
for field in ("v_call", "d_call", "j_call"):
    chains[field] = ak.str.replace_substring_regex(chains[field], r"\*\d+", "")
vdjdb.obsm["airr"] = chains
vdjdb

SHA256 hash of downloaded file: 1d1b3e7e3b97dcf817da18803217e50e3893439d7e62bfa66b1fad54629f09dc
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.


AnnData object with n_obs × n_vars = 175339 × 0
    obs: 'MHC_class', 'MHC_gene_1', 'MHC_gene_2', 'PMID', 'antigen_name', 'antigen_species', 'assay_category', 'assay_method_raw', 'epitope_sequence', 'iedb_iri', 'source_organism', 'tissue', 'source'
    uns: 'DB', 'chain_indices', 'iggytop_metadata'
    obsm: 'airr', 'chain_indices'

In [8]:
ln.Artifact.from_anndata(
    vdjdb,
    key="air_repertoire/vdjdb_human.h5ad",
    description="Human TCRs of VDJdb from IggyTop data-2026.09.28.152145 without gene alleles",
).save()
ln.Artifact(
    cov_abdab_path,
    key="air_repertoire/cov_abdab_080224.csv",
    description="CoV-AbDab release of 8 February 2024",
).save()

... uploading V00dDz5lHdI4QNrD0000.h5ad:  0.0%

... uploading V00dDz5lHdI4QNrD0000.h5ad: 100.0%
! data is a DataFrame, please use .from_dataframe()


... uploading 79d0a0633ed15bcb1852d184aa1e7f69-CoV-AbDab_080224.csv:  0.0%

... uploading 79d0a0633ed15bcb1852d184aa1e7f69-CoV-AbDab_080224.csv: 100.0%


Artifact(uid='HOiyuPWjtHbWKAyG0000', key='air_repertoire/cov_abdab_080224.csv', description='CoV-AbDab release of 8 February 2024', suffix='.csv', kind=None, otype='DataFrame', size=8467893, hash='S8vOw_NbwM-3JTW7zz3_CA', n_files=None, n_observations=None, extra_data=None, branch_id=1, created_on_id=1, space_id=1, storage_id=1, run_id=175, schema_id=None, created_by_id=2, created_at=2026-09-29 16:58:17 UTC, is_locked=False, version_tag=None, is_latest=True)

In [9]:
ln.finish()